<h1 style="font-size: 40px;">Gravitaional Waves DT Training </h1>

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingRegressor, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt


features = pd.read_csv("features.csv", index_col=0)  # feature} columns only
targets = pd.read_csv("targets.csv", index_col=0)  
features = features.loc[targets.index]
feature_names = features.columns
targets=y = targets.values.ravel()


<h1 style="font-size: 32px;">One Decision Tree Model  </h1>

In [4]:
dt_model=DecisionTreeClassifier(min_samples_split=10)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
scores = cross_validate(dt_model, features_scaled, targets, cv = StratifiedKFold(n_splits=5), return_train_score=True)
scores


{'fit_time': array([0.13655829, 0.12705898, 0.17698073, 0.11037397, 0.0989666 ]),
 'score_time': array([0.00173211, 0.00150108, 0.0019033 , 0.0021925 , 0.00185013]),
 'test_score': array([0.5015, 0.501 , 0.4865, 0.515 , 0.488 ]),
 'train_score': array([0.872125, 0.877125, 0.878   , 0.881   , 0.878625])}

Feature Importance 

In [13]:

dt_model.fit(features_scaled, targets)
importances = dt_model.feature_importances_
feature_names = (
    features.columns 
    if hasattr(features, "columns") 
    else [f"feature_{i}" for i in range(features_scaled.shape[1])]
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
2,correlated_snr,0.260287
3,chi2,0.256661
1,cross_correlation,0.242580
0,norm,0.240472
4,snr_likelihood,0.000000


<h1 style="font-size: 32px;">Random Forest (Bagged Trees)  </h1>

In [5]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, min_samples_split=10)
scores = cross_validate(rf_model, features_scaled, targets, cv = StratifiedKFold(n_splits=5), return_train_score=True)
scores


{'fit_time': array([2.58195662, 2.86223364, 2.64526486, 2.4462142 , 2.56028533]),
 'score_time': array([0.034513  , 0.03786111, 0.03907013, 0.04455853, 0.04218984]),
 'test_score': array([0.505 , 0.5035, 0.4915, 0.5   , 0.4825]),
 'train_score': array([0.991875, 0.989   , 0.99    , 0.990375, 0.98925 ])}

Feature Importance

In [12]:

rf_model.fit(features_scaled, targets)
importances = rf_model.feature_importances_
feature_names = (
    features.columns 
    if hasattr(features, "columns") 
    else [f"feature_{i}" for i in range(features_scaled.shape[1])]
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
2,correlated_snr,0.255389
1,cross_correlation,0.252130
3,chi2,0.248423
0,norm,0.244059
4,snr_likelihood,0.000000


<h1 style="font-size: 32px;">Gradient Boosted Trees  </h1>

In [8]:
gb_model= GradientBoostingClassifier(max_depth=10, n_estimators=10)
scores = cross_validate(gb_model, features_scaled, targets, cv = StratifiedKFold(n_splits=5), return_train_score=True)
scores



{'fit_time': array([0.67178607, 0.58807063, 0.58328843, 0.57111168, 0.55862689]),
 'score_time': array([0.00227594, 0.00211549, 0.00204515, 0.00172448, 0.00169754]),
 'test_score': array([0.4845, 0.5045, 0.524 , 0.502 , 0.4885]),
 'train_score': array([0.72025 , 0.655375, 0.656625, 0.724125, 0.656625])}

Increasing the Number of Estimators 

In [9]:
gb_model= GradientBoostingClassifier(max_depth=10, n_estimators=100)
scores = cross_validate(gb_model, features_scaled, targets, cv = StratifiedKFold(n_splits=5), return_train_score=True)
scores

{'fit_time': array([5.56443691, 5.72747874, 5.40908527, 5.58754611, 5.29914713]),
 'score_time': array([0.0080142 , 0.00718236, 0.00833583, 0.00884175, 0.00644302]),
 'test_score': array([0.488 , 0.5055, 0.515 , 0.516 , 0.487 ]),
 'train_score': array([0.962625, 0.954125, 0.95075 , 0.96175 , 0.96025 ])}

Feature Importance

In [11]:

gb_model.fit(features_scaled, targets)
importances = gb_model.feature_importances_
feature_names = (
    features.columns 
    if hasattr(features, "columns") 
    else [f"feature_{i}" for i in range(features_scaled.shape[1])]
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df



,feature,importance
2,correlated_snr,0.263450
1,cross_correlation,0.256799
3,chi2,0.250373
0,norm,0.229378
4,snr_likelihood,0.000000


Hyperparamter Search of Gradient Boosted Trees 

In [64]:
pd.set_option("display.max_rows", None)         # show all rows
pd.set_option("display.max_columns", None)      # show all columns
pd.set_option("display.max_colwidth", None)     # show full text in each column
pd.set_option("display.expand_frame_repr", True)  # don't wrap to multiple lines



parameters = {'max_depth':[6,10,15,None], 
              'max_iter':[100,200,500], 'learning_rate': [0.05, 0.1,0.3,0.5], 
              'early_stopping':[True, False]}

randomsearch = GridSearchCV(HistGradientBoostingClassifier(), parameters, cv = StratifiedKFold(n_splits=3, shuffle=True), \
                     verbose = 2, n_jobs = 4, return_train_score=True)
randomsearch.fit(features_scaled, targets)

scores = pd.DataFrame(randomsearch.cv_results_)
scoresCV = scores[['params','mean_test_score','std_test_score','mean_train_score']].sort_values(by = 'mean_test_score', \
                                                    ascending = False)
scoresCV

Fitting 3 folds for each of 96 candidates, totalling 288 fits


,params,mean_test_score,std_test_score,mean_train_score
28,"{'early_stopping': True, 'learning_rate': 0.3, 'max_depth': 10, 'max_iter': 200}",0.510300,0.005395,0.653850
6,"{'early_stopping': True, 'learning_rate': 0.05, 'max_depth': 15, 'max_iter': 100}",0.509299,0.008808,0.604600
3,"{'early_stopping': True, 'learning_rate': 0.05, 'max_depth': 10, 'max_iter': 100}",0.508998,0.011271,0.598800
30,"{'early_stopping': True, 'learning_rate': 0.3, 'max_depth': 15, 'max_iter': 100}",0.508300,0.004734,0.645150
26,"{'early_stopping': True, 'learning_rate': 0.3, 'max_depth': 6, 'max_iter': 500}",0.507900,0.004982,0.596650
8,"{'early_stopping': True, 'learning_rate': 0.05, 'max_depth': 15, 'max_iter': 500}",0.507799,0.004960,0.619902
16,"{'early_stopping': True, 'learning_rate': 0.1, 'max_depth': 10, 'max_iter': 200}",0.507401,0.007363,0.616000
10,"{'early_stopping': True, 'learning_rate': 0.05, 'max_depth': None, 'max_iter': 200}",0.507200,0.001829,0.629550
67,"{'early_stopping': False, 'learning_rate': 0.1, 'max_depth': 15, 'max_iter': 200}",0.506899,0.008460,0.850700
46,"{'early_stopping': True, 'learning_rate': 0.5, 'max_depth': None, 'max_iter': 200}",0.506800,0.003360,0.656850
